# 00 · Consolidación de datos — Pruebas APRENDER y Estadística Educativa

Este notebook consolida **todos** los archivos de `resultados_aprender/` en un dataset
analítico documentado, guardado en `datos_consolidados/`.

## Qué produce

| Salida | Contenido |
|---|---|
| `catalogo_archivos.csv` | Un registro por archivo fuente con sus atributos parseados |
| `diccionario_maestro.xlsx` | Diccionario unificado (RA + APRENDER) + catálogo + notas de unidades |
| `diccionario_ra.parquet` / `diccionario_aprender.parquet` | Diccionarios en formato tabular |
| `ra/ra_<base>.parquet` (7) | Familia RA (Relevamiento Anual) en formato **ancho**, todos los años apilados |
| `aprender_long/anio=YYYY/*.parquet` | Familia APRENDER en formato **largo/tidy**, particionado por año |

## Modelo de datos y decisiones

Los datos son **dos familias con granularidades distintas**, por eso no se fuerzan en una sola tabla:

- **RA (estadística educativa)** — 7 sub-bases × 15 años (2011–2025). Grano: `provincia × departamento
  × sector × ámbito` (+ claves extra en *Cargos Bis* y *Matrícula por edad*). Columnas = medidas bien
  definidas → se conserva **formato ancho** (una columna por variable), apilando años con la columna `anio`.
  Valores = **conteos absolutos** (matrícula, cargos, población, etc.).
- **APRENDER (evaluación)** — 48 archivos (2016–2025, sin 2020). Miles de columnas `pregunta_opción`
  que **cambian año a año**, por lo que la única forma consistente de unificarlas es **formato largo/tidy**.
  Valores = **conteos ponderados de estudiantes** (factor de expansión) por opción de respuesta.

> **Por qué Parquet y no un único Excel:** varias tablas superan el límite de Excel (1.048.576 filas;
> p. ej. *Cargos Bis* tiene 1,46 M de filas y APRENDER largo ~19 M). Parquet es comprimido, tipado y
> se lee particionado. El **diccionario maestro** sí se entrega en Excel legible.

> **Nivel/grado de los archivos 2016–2018:** varios nombres de archivo no traen el nivel y/o el grado. Se
> completan con el operativo real (2016 y 2017: Primaria **6 grado** y Secundaria **5-6 año**; 2018: Primaria
> **6 grado**), verificado con la **edad más frecuente** que declaran los estudiantes (11 años → 6° grado;
> 17 años → 5-6° año). La sección 8 lo controla automáticamente y el catálogo marca esos archivos en
> `nivel_grado_inferido`.

El significado de cada variable se resuelve por *join* con el **diccionario maestro** (no se duplica el
texto de las preguntas en la tabla de datos, que se mantiene liviana).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]
(https://colab.research.google.com/github/santiagoriverti/analisis_pruebas_aprender/blob/main/00_consolidacion.ipynb)

---
## 0 · Bootstrap para Google Colab

En Colab se clona el repositorio (para tener `resultados_aprender/`) y se instalan las dependencias.
Fuera de Colab esta celda no hace nada. Corré **Entorno de ejecución → Ejecutar todo**.

In [1]:
import os, sys, subprocess

REPO_URL = 'https://github.com/santiagoriverti/analisis_pruebas_aprender.git'
REPO_DIR = 'analisis_pruebas_aprender'

def in_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False

if in_colab():
    dest = os.path.join('/content', REPO_DIR)
    if not os.path.isdir(dest):
        print('Clonando el repositorio (incluye ~520 MB de .xlsx, puede tardar 1-3 min)...')
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, dest], check=True)
    os.chdir(dest)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-calamine', 'pyarrow'], check=True)
    print('Colab listo. Directorio de trabajo:', os.getcwd())
else:
    print('Entorno local (no Colab). Directorio de trabajo:', os.getcwd())

Entorno local (no Colab). Directorio de trabajo: C:\Users\sriverti\Desktop\INECO\Repositorios\analisis_pruebas_aprender


---
## 1 · Configuración e imports

In [2]:
import os, re, sys, time, subprocess
import pandas as pd
import numpy as np

# Motor de lectura rápido para .xlsx (5-10x más veloz que openpyxl).
try:
    import python_calamine  # noqa
    ENGINE = 'calamine'
except ImportError:
    print('Instalando python-calamine...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-calamine'], check=True)
    ENGINE = 'calamine'

# Rutas (relativas a la raíz del repo, donde vive este notebook).
REPO = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
DATA = os.path.join(REPO, 'resultados_aprender')
OUT  = os.path.join(REPO, 'datos_consolidados')
os.makedirs(os.path.join(OUT, 'ra'), exist_ok=True)
os.makedirs(os.path.join(OUT, 'aprender_long'), exist_ok=True)

xlsx = sorted(f for f in os.listdir(DATA) if f.lower().endswith('.xlsx'))
print(f'{len(xlsx)} archivos .xlsx en resultados_aprender/  | motor de lectura: {ENGINE}')

156 archivos .xlsx en resultados_aprender/  | motor de lectura: calamine


---
## 2 · Funciones de parsing y consolidación

In [3]:
RA_BASES = ['Matricula por edad', 'Matricula', 'Cargos Bis', 'Cargos',
            'Caracteristicas', 'Poblacion', 'Trayectoria']  # 'por edad' antes que 'Matricula'
GEO_KEYS = ['provincia', 'departamento', 'sector', 'ambito']
RA_EXTRA_KEYS = {'Cargos Bis': ['nivel', 'tipo', 'categoria', 'cargo', 'planta_tipo'],
                 'Matricula por edad': ['grado']}
DESEMP_RE = re.compile(r'^(l|m|cn|cs|c)desemp', re.I)
AP_RE     = re.compile(r'^ap\d+[a-z]?\d*', re.I)   # ap01, ap18a, ap20b1, Ap1...

def anio_de(f):
    m = re.match(r'(\d{4})', f); return int(m.group(1)) if m else None

def clasificar(f):
    if 'Diccionario' in f: return ('DICCIONARIO', None)
    if f.startswith('ICSE'): return ('OTRO', None)
    if 'Base APRENDER' in f: return ('APRENDER', None)
    for b in RA_BASES:
        if f'{b} - agregada' in f: return ('RA', b)
    return ('DESCONOCIDO', None)

# Archivos 2016–2018 cuyo nombre no trae nivel y/o grado: se completan con el operativo real,
# verificado con la edad más frecuente declarada por los estudiantes (control en la sección 8):
# 11 años -> Primaria 6° grado; 17 años -> Secundaria 5-6° año.
NIVEL_SIN_NOMBRE = {2018: 'Primaria'}
GRADO_SIN_NOMBRE = {(2016, 'Primaria'): '6 grado', (2016, 'Secundaria'): '5-6 año',
                    (2017, 'Primaria'): '6 grado', (2017, 'Secundaria'): '5-6 año',
                    (2018, 'Primaria'): '6 grado'}

def parse_nombre_aprender(f):
    """Atributos tal como figuran en el nombre del archivo (None si no aparecen)."""
    low = f.lower()
    cob = 'Muestral' if 'muestra' in low else 'Censal'
    niv = 'Primaria' if 'primaria' in low else ('Secundaria' if 'secundaria' in low else None)
    gm = re.search(r'(\d\s*-\s*\d|\d)\s*(grado|a[nñ]o)', low)
    grado = None
    if gm:  # normaliza '3grado' -> '3 grado', '5 - 6 año' -> '5-6 año'
        grado = re.sub(r'\s+', '', gm.group(1)) + ' ' + ('grado' if gm.group(2) == 'grado' else 'año')
    if 'solo cc' in low:
        area = 'Solo CC'
    else:
        am = re.search(r'desempe[nñ]os de ([a-záéíóúñ ]+?)\.xlsx', low)
        area = am.group(1).strip().title() if am else 'No especificado'
    return dict(cobertura=cob, nivel=niv, grado=grado, area=area)

def parse_aprender(f):
    meta = parse_nombre_aprender(f); anio = anio_de(f)
    if meta['nivel'] is None:
        meta['nivel'] = NIVEL_SIN_NOMBRE.get(anio, 'No especificado')
    if meta['grado'] is None:
        meta['grado'] = GRADO_SIN_NOMBRE.get((anio, meta['nivel']), 'No especificado')
    return meta

def tipo_variable(col):
    if DESEMP_RE.match(col): return 'desempeño'
    if AP_RE.match(col):     return 'contexto'
    if col.startswith('NSE'): return 'nse'
    if col.startswith('Nivel_Ed'): return 'nivel_educativo_hogar'
    return 'otro'

def _norm_keys(df):
    ren = {}
    for c in df.columns:
        low = str(c).strip().lower()
        if low == 'provincia': ren[c] = 'provincia'
        elif low == 'departamento': ren[c] = 'departamento'
        elif low in ('cod_provincia', 'jurisdiccion', 'jurisdicción'): ren[c] = 'jurisdiccion'
        elif low == 'sector': ren[c] = 'sector'
        elif low in ('ambito', 'ámbito'): ren[c] = 'ambito'
    return df.rename(columns=ren)

def leer_in(path):
    return pd.read_excel(path, sheet_name='in', engine=ENGINE)

def aprender_to_long(path):
    f = os.path.basename(path)
    meta = parse_aprender(f); anio = anio_de(f)
    df = _norm_keys(leer_in(path))
    keys = [k for k in ['jurisdiccion', 'departamento', 'sector', 'ambito'] if k in df.columns]
    valcols = [c for c in df.columns if c not in keys]
    long = df.melt(id_vars=keys, value_vars=valcols, var_name='variable', value_name='valor')
    long['valor'] = pd.to_numeric(long['valor'], errors='coerce')
    long = long.dropna(subset=['valor'])          # celdas vacías = no aplica
    long['anio'] = anio
    for k, v in meta.items(): long[k] = v
    if 'departamento' not in long.columns: long['departamento'] = pd.NA
    long['tipo_variable'] = long['variable'].map(tipo_variable)
    return long[['anio','cobertura','nivel','grado','area','jurisdiccion','departamento',
                 'sector','ambito','variable','tipo_variable','valor']]

def ra_base_wide(base, files):
    frames = []
    for f in files:
        df = _norm_keys(leer_in(os.path.join(DATA, f)))
        df.insert(0, 'anio', anio_de(f))
        frames.append(df)
    out = pd.concat(frames, ignore_index=True, sort=False)
    keys = ['anio'] + [k for k in GEO_KEYS if k in out.columns] + \
           [k for k in RA_EXTRA_KEYS.get(base, []) if k in out.columns]
    rest = [c for c in out.columns if c not in keys]
    return out[keys + rest]

def coerce_for_parquet(df, keys):
    """Garantiza tipos homogéneos por columna para escribir Parquet sin errores."""
    for c in df.columns:
        if c == 'anio':
            df[c] = df[c].astype('int16'); continue
        if c in keys:
            df[c] = df[c].astype('string'); continue
        if df[c].dtype == object:
            num = pd.to_numeric(df[c], errors='coerce')
            nn = df[c].notna().sum()
            df[c] = num if (nn > 0 and num.notna().sum() >= 0.95 * nn) else df[c].astype('string')
    return df
print('Funciones definidas.')

Funciones definidas.


---
## 3 · Catálogo de archivos fuente

In [4]:
cat_rows = []
for f in xlsx:
    fam, base = clasificar(f)
    row = {'archivo': f, 'familia': fam, 'base': base, 'anio': anio_de(f)}
    if fam == 'APRENDER':
        row.update(parse_aprender(f))
        crudo = parse_nombre_aprender(f)
        row['nivel_grado_inferido'] = crudo['nivel'] is None or crudo['grado'] is None
    cat_rows.append(row)
catalogo = pd.DataFrame(cat_rows)
print(catalogo.familia.value_counts().to_dict())
_ap = catalogo[catalogo.familia == 'APRENDER']
assert not _ap[['nivel', 'grado']].isin(['No especificado']).any().any(), 'Hay archivos APRENDER sin nivel/grado'
print('Archivos APRENDER con nivel/grado completado (no figura en el nombre):', int(_ap.nivel_grado_inferido.sum()))
catalogo.head(10)

{'RA': 105, 'APRENDER': 48, 'DICCIONARIO': 2, 'OTRO': 1}
Archivos APRENDER con nivel/grado completado (no figura en el nombre): 17


,archivo,familia,base,anio,cobertura,nivel,grado,area,nivel_grado_inferido
0,2011 - 2024 - Diccionario bases aprender anon...,DICCIONARIO,None,2011.0,NaN,NaN,NaN,NaN,NaN
1,2011 Caracteristicas - agregada.xlsx,RA,Caracteristicas,2011.0,NaN,NaN,NaN,NaN,NaN
2,2011 Cargos - agregada.xlsx,RA,Cargos,2011.0,NaN,NaN,NaN,NaN,NaN
3,2011 Cargos Bis - agregada.xlsx,RA,Cargos Bis,2011.0,NaN,NaN,NaN,NaN,NaN
4,2011 Matricula - agregada.xlsx,RA,Matricula,2011.0,NaN,NaN,NaN,NaN,NaN
5,2011 Matricula por edad - agregada.xlsx,RA,Matricula por edad,2011.0,NaN,NaN,NaN,NaN,NaN
6,2011 Poblacion - agregada.xlsx,RA,Poblacion,2011.0,NaN,NaN,NaN,NaN,NaN
7,2011 Trayectoria - agregada.xlsx,RA,Trayectoria,2011.0,NaN,NaN,NaN,NaN,NaN
8,2012 Caracteristicas - agregada.xlsx,RA,Caracteristicas,2012.0,NaN,NaN,NaN,NaN,NaN
9,2012 Cargos - agregada.xlsx,RA,Cargos,2012.0,NaN,NaN,NaN,NaN,NaN


---
## 4 · Diccionario maestro
Se combinan las dos fuentes oficiales. Para APRENDER, cada columna `pregunta_opción` se mapea a su
etiqueta `"<pregunta> - <opción>"`, que se separa en `pregunta_texto` y `opcion_texto`.

In [5]:
# 4a · Diccionario RA (campo | tipo | contenido) por base
ra_dic_wb = pd.read_excel(os.path.join(DATA, 'Diccionario de bases RA anonimizadas.xlsx'),
                          sheet_name=None, engine=ENGINE, header=None)
frames = []
for sh, df in ra_dic_wb.items():
    if sh == 'Presentacion': continue
    d = df.iloc[1:, :3].copy(); d.columns = ['campo', 'tipo_campo', 'contenido']
    d.insert(0, 'base', sh); frames.append(d)
dic_ra = pd.concat(frames, ignore_index=True).dropna(subset=['campo'])

# 4b · Diccionario APRENDER (columna -> pregunta/opción) por hoja (año+nivel)
ap_dic_wb = pd.read_excel(os.path.join(DATA, '2011 - 2024  - Diccionario bases aprender anonimizadas.xlsx'),
                          sheet_name=None, engine=ENGINE, header=None)
frames = []
for sh, df in ap_dic_wb.items():
    if sh == 'Presentacion': continue
    d = df.iloc[2:, :2].copy(); d.columns = ['variable', 'etiqueta']
    d = d.dropna(subset=['variable']); d.insert(0, 'hoja_dic', sh)
    m = re.match(r'(\d{4})', str(sh)); d['anio_dic'] = int(m.group(1)) if m else None
    low = str(sh).lower()
    d['nivel_dic'] = 'Primaria' if 'pri' in low else ('Secundaria' if 'sec' in low else 'General')
    et = d['etiqueta'].astype(str)
    d['pregunta_texto'] = et.str.rsplit(' - ', n=1).str[0]
    d['opcion_texto']   = et.str.rsplit(' - ', n=1).str[-1]
    frames.append(d)
dic_ap = pd.concat(frames, ignore_index=True)
print('dic_ra:', dic_ra.shape, '| dic_ap:', dic_ap.shape)
dic_ap[dic_ap.variable.isin(['ap03_Masculino','ldesemp_Básico'])][['hoja_dic','variable','pregunta_texto','opcion_texto']].head()

dic_ra: (701, 4) | dic_ap: (10775, 7)


,hoja_dic,variable,pregunta_texto,opcion_texto
320,2016 Estudiantes - APRENDER Pri,ldesemp_Básico,Nivel de Desempeño en Lengua,Básico
444,2016 Muestra APRENDER - Pri,ldesemp_Básico,Nivel de Desempeño en Lengua,Básico
1023,2016 Estudiantes - APRENDER Sec,ldesemp_Básico,Nivel de Desempeño en Lengua,Básico
1608,2016 Muestra APRENDER - Sec,ldesemp_Básico,Nivel de Desempeño en Lengua,Básico
3398,2017 Estudiantes - APRENDER Sec,ldesemp_Básico,Nivel de desempeño en Lengua,Básico


---
## 5 · Familia RA → 7 Parquet anchos

In [6]:
ra_index = []
for base in RA_BASES:
    files = sorted(f for f in xlsx if clasificar(f) == ('RA', base))
    w = ra_base_wide(base, files)
    keys = ['anio'] + [k for k in GEO_KEYS if k in w.columns] + \
           [k for k in RA_EXTRA_KEYS.get(base, []) if k in w.columns]
    w = coerce_for_parquet(w, keys)
    safe = base.lower().replace(' ', '_')
    p = os.path.join(OUT, 'ra', f'ra_{safe}.parquet')
    w.to_parquet(p, engine='pyarrow', compression='zstd', index=False)
    ra_index.append({'base': base, 'archivo_parquet': f'ra/ra_{safe}.parquet',
                     'filas': len(w), 'columnas': w.shape[1], 'MB': round(os.path.getsize(p)/1e6, 2)})
    print(f'  {base:20s} -> {w.shape}')
ra_index = pd.DataFrame(ra_index); ra_index

  Matricula por edad   -> (300333, 44)


  Matricula            -> (18020, 105)


  Cargos Bis           -> (1464119, 11)


  Cargos               -> (18426, 61)


  Caracteristicas      -> (18021, 74)


  Poblacion            -> (18017, 170)


  Trayectoria          -> (17912, 317)


,base,archivo_parquet,filas,columnas,MB
0,Matricula por edad,ra/ra_matricula_por_edad.parquet,300333,44,2.85
1,Matricula,ra/ra_matricula.parquet,18020,105,1.91
2,Cargos Bis,ra/ra_cargos_bis.parquet,1464119,11,3.75
3,Cargos,ra/ra_cargos.parquet,18426,61,0.68
4,Caracteristicas,ra/ra_caracteristicas.parquet,18021,74,0.68
5,Poblacion,ra/ra_poblacion.parquet,18017,170,1.10
6,Trayectoria,ra/ra_trayectoria.parquet,17912,317,4.46


---
## 6 · Familia APRENDER → Parquet largo particionado por año
Cada archivo se transforma a formato largo y se escribe en `aprender_long/anio=YYYY/`.
Se procesa de a un archivo (memoria acotada).

In [7]:
ap_files = sorted(f for f in xlsx if clasificar(f)[0] == 'APRENDER')
ap_index, tot = [], 0
CATCOLS = ['cobertura','nivel','grado','area','tipo_variable','jurisdiccion','sector','ambito','departamento','variable']
for f in ap_files:
    lg = aprender_to_long(os.path.join(DATA, f))
    for c in CATCOLS: lg[c] = lg[c].astype('string')
    anio = int(lg['anio'].iloc[0])
    d = os.path.join(OUT, 'aprender_long', f'anio={anio}'); os.makedirs(d, exist_ok=True)
    safe = re.sub(r'[^0-9A-Za-z]+', '_', f)[:80]
    lg.drop(columns=['anio']).to_parquet(os.path.join(d, safe + '.parquet'),
                                          engine='pyarrow', compression='zstd', index=False)
    tot += len(lg); ap_index.append({'archivo': f, 'anio': anio, 'filas': len(lg)})
ap_index = pd.DataFrame(ap_index)
print(f'APRENDER largo: {tot:,} filas en {len(ap_index)} archivos')
ap_index.groupby('anio').filas.sum()

APRENDER largo: 18,721,597 filas en 48 archivos


anio
2016    3788601
2017    2971268
2018    1079294
2019    2333376
2021    1740100
2022    2175201
2023    1776906
2024    1711855
2025    1144996
Name: filas, dtype: int64

---
## 7 · Diccionario maestro Excel + catálogos

In [8]:
notas = pd.DataFrame({
  'tema': ['Unidades APRENDER','Unidades RA','Formato','Claves geo','Valores 0/vacíos',
           'Desempeño','Cobertura','Nivel/grado 2016-2018','Comparabilidad','Fuente'],
  'detalle': [
   'Valores APRENDER agregado = CONTEOS PONDERADOS de estudiantes (factor de expansión) por opción de respuesta.',
   'Bases RA = CONTEOS ABSOLUTOS (matrícula, cargos, población) agregados a depto/sector/ámbito.',
   'Datos en Parquet (zstd). Excel no alcanza: varias tablas superan 1.048.576 filas.',
   'provincia/jurisdiccion, departamento (puede faltar: agregado a nivel provincia), sector (Estatal/Privado), ambito (Urbano/Rural).',
   'APRENDER largo descarta celdas vacías (no aplica). RA ancho: vacío = sin dato para esa columna/año.',
   'Niveles: Por debajo del básico / Básico / Satisfactorio / Avanzado (2024 Prim 3°: Lector incipiente / Nivel I-V). Prefijos l=Lengua, m=Matemática, cn=Cs Naturales, cs=Cs Sociales, c=Ciudadanía.',
   'Censal (universo) vs Muestral. 2016-2018 sin la palabra en el nombre = operativo principal (Censal).',
   'Archivos sin nivel/grado en el nombre: 2016 y 2017 Primaria = 6 grado, Secundaria = 5-6 año; 2018 = Primaria 6 grado. Verificado con la edad modal declarada (11 / 17 años). Ver columna nivel_grado_inferido del catálogo.',
   'El operativo cambia de nivel/grado/cobertura por año; no todas las series son directamente comparables.',
   'Secretaría de Educación de la Nación (Argentina) - datos abiertos.']})

with pd.ExcelWriter(os.path.join(OUT, 'diccionario_maestro.xlsx'), engine='openpyxl') as xw:
    notas.to_excel(xw, sheet_name='Notas', index=False)
    catalogo.to_excel(xw, sheet_name='Catalogo', index=False)
    dic_ra.to_excel(xw, sheet_name='Diccionario_RA', index=False)
    dic_ap.to_excel(xw, sheet_name='Diccionario_APRENDER', index=False)
    ra_index.to_excel(xw, sheet_name='Indice_RA', index=False)
    ap_index.to_excel(xw, sheet_name='Indice_APRENDER', index=False)

catalogo.to_csv(os.path.join(OUT, 'catalogo_archivos.csv'), index=False, encoding='utf-8-sig')
dic_ra.to_parquet(os.path.join(OUT, 'diccionario_ra.parquet'), index=False)
dic_ap.to_parquet(os.path.join(OUT, 'diccionario_aprender.parquet'), index=False)
print('Diccionario maestro y catálogos escritos.')

Diccionario maestro y catálogos escritos.


---
## 8 · Validaciones y resumen

In [9]:
# Re-lectura de una partición y de una base RA
s = pd.read_parquet(os.path.join(OUT, 'aprender_long', 'anio=2024'))
print('APRENDER 2024:', s.shape, '| áreas:', sorted(s.area.dropna().unique()))
print('tipo_variable:', s.tipo_variable.value_counts().to_dict())
rm = pd.read_parquet(os.path.join(OUT, 'ra', 'ra_matricula.parquet'))
print('RA matrícula:', rm.shape, '| años:', rm.anio.min(), '-', rm.anio.max())

def dirsize(p):
    return sum(os.path.getsize(os.path.join(r, x)) for r, _, fs in os.walk(p) for x in fs) / 1e6
print(f'Tamaño total datos_consolidados/: {dirsize(OUT):.1f} MB')

# Control de nivel/grado: la edad más frecuente declarada por los estudiantes (pregunta 1 del cuestionario)
# debe ser la edad típica del grado asignado. Cubre los archivos 2016–2018 (incluidos los de grado completado).
EDAD_TIPICA = {'3 grado': 8, '6 grado': 11, '2-3 año': 14, '5-6 año': 17}
filas_edad = []
for anio in [2016, 2017, 2018]:
    d = pd.read_parquet(os.path.join(OUT, 'aprender_long', f'anio={anio}'),
                        columns=['cobertura', 'nivel', 'grado', 'variable', 'valor'])
    d = d[d.variable.str.match(r'^ap0?1_', case=False)]
    d = d.assign(edad=d.variable.str.extract(r'(\d+)_a[ñn]os', expand=False)).dropna(subset=['edad'])
    for (cob, niv, gra), g in d.groupby(['cobertura', 'nivel', 'grado']):
        filas_edad.append({'anio': anio, 'cobertura': cob, 'nivel': niv, 'grado': gra,
                           'edad_modal': int(g.groupby('edad').valor.sum().idxmax()),
                           'edad_tipica_grado': EDAD_TIPICA.get(gra)})
control_grado = pd.DataFrame(filas_edad)
assert (control_grado.edad_modal == control_grado.edad_tipica_grado).all(), control_grado
print('Control de nivel/grado 2016–2018 (edad modal = edad típica del grado): OK')
control_grado

APRENDER 2024: (1711855, 11) | áreas: ['Lengua', 'Matematica', 'Solo CC']
tipo_variable: {'contexto': 1567592, 'nivel_educativo_hogar': 62083, 'otro': 55906, 'nse': 17620, 'desempeño': 8654}
RA matrícula: (18020, 105) | años: 2011 - 2025
Tamaño total datos_consolidados/: 171.5 MB


Control de nivel/grado 2016–2018 (edad modal = edad típica del grado): OK


,anio,cobertura,nivel,grado,edad_modal,edad_tipica_grado
0,2016,Censal,Primaria,6 grado,11,11
1,2016,Censal,Secundaria,5-6 año,17,17
2,2016,Muestral,Primaria,3 grado,8,8
3,2016,Muestral,Secundaria,2-3 año,14,14
4,2017,Censal,Primaria,6 grado,11,11
5,2017,Censal,Secundaria,5-6 año,17,17
6,2018,Censal,Primaria,6 grado,11,11


---
## 9 · Verificación (bloque para compartir)
Ejecutá esta celda y **copiá TODO el bloque** entre las líneas `=== ...`. La `firma` resume las
métricas de datos (invariantes del entorno): si coincide con la de referencia, la consolidación
compiló idéntica y sin errores. Incluye los operativos APRENDER `(año, cobertura, nivel, grado)`.

In [10]:
import hashlib, platform
from importlib.metadata import version, PackageNotFoundError
def _ver(pkg):
    try: return version(pkg)
    except PackageNotFoundError: return 'NO'
_pa, _cal = _ver('pyarrow'), _ver('python-calamine')

# Métricas de datos (deben ser idénticas en cualquier entorno)
cb = pd.read_parquet(os.path.join(OUT, 'ra', 'ra_cargos_bis.parquet'))
s24 = pd.read_parquet(os.path.join(OUT, 'aprender_long', 'anio=2024'))
data_lines = [
    f'archivos_por_familia={dict(sorted(catalogo.familia.value_counts().to_dict().items()))}',
    f'dic_ra={tuple(dic_ra.shape)} dic_ap={tuple(dic_ap.shape)}',
    'operativos=' + str(sorted({(int(r.anio), r.cobertura, r.nivel, r.grado)
                               for r in catalogo[catalogo.familia == 'APRENDER'].itertuples()})),
] + [f'RA[{r.base}]=filas:{r.filas},cols:{r.columnas}' for _, r in ra_index.iterrows()] + [
    f'APRENDER_total_filas={int(ap_index.filas.sum())}',
    f'APRENDER_por_anio={dict(sorted(ap_index.groupby("anio").filas.sum().to_dict().items()))}',
    f'cargos_bis_total_2024={cb.loc[cb.anio==2024, "total"].astype("float64").sum():.1f}',
    f'tipo_variable_2024={dict(sorted(s24.tipo_variable.value_counts().to_dict().items()))}',
]
firma = hashlib.sha256('|'.join(data_lines).encode('utf-8')).hexdigest()[:12]

def _dirsize(p):
    return sum(os.path.getsize(os.path.join(r, x)) for r, _, fs in os.walk(p) for x in fs) / 1e6
env_lines = [
    f'python={platform.python_version()} pandas={pd.__version__} pyarrow={_pa} calamine={_cal}',
    f'colab={in_colab()} salida_MB={_dirsize(OUT):.1f}',
]

print('=== VERIFICACION 00_consolidacion — copiar TODO este bloque ===')
for l in env_lines + data_lines: print(l)
print(f'firma={firma}')
print('=== fin verificacion ===')

=== VERIFICACION 00_consolidacion — copiar TODO este bloque ===
python=3.14.5 pandas=2.3.3 pyarrow=24.0.0 calamine=0.8.2
colab=False salida_MB=171.5
archivos_por_familia={'APRENDER': 48, 'DICCIONARIO': 2, 'OTRO': 1, 'RA': 105}
dic_ra=(701, 4) dic_ap=(10775, 7)
operativos=[(2016, 'Censal', 'Primaria', '6 grado'), (2016, 'Censal', 'Secundaria', '5-6 año'), (2016, 'Muestral', 'Primaria', '3 grado'), (2016, 'Muestral', 'Secundaria', '2-3 año'), (2017, 'Censal', 'Primaria', '6 grado'), (2017, 'Censal', 'Secundaria', '5-6 año'), (2018, 'Censal', 'Primaria', '6 grado'), (2019, 'Censal', 'Secundaria', '5-6 año'), (2019, 'Muestral', 'Secundaria', '5-6 año'), (2021, 'Censal', 'Primaria', '6 grado'), (2022, 'Censal', 'Secundaria', '5-6 año'), (2022, 'Muestral', 'Primaria', '6 grado'), (2023, 'Censal', 'Primaria', '6 grado'), (2024, 'Censal', 'Secundaria', '5-6 año'), (2024, 'Muestral', 'Primaria', '3 grado'), (2025, 'Censal', 'Primaria', '6 grado')]
RA[Matricula por edad]=filas:300333,cols:44
RA[

---
## 10 · Guardar en Google Drive (Mi unidad/pruebas_aprender)
Al ejecutar en **Colab**, esta celda monta tu Google Drive, crea la carpeta `pruebas_aprender` en
*Mi unidad* y guarda los resultados en dos formatos:

- **`parquet/`** — formato eficiente/tipado, pensado para que el **notebook 01** procese los datos.
- **`excel/`** — legible por humanos: `.xlsx` para las tablas navegables (≤200k filas) y `.csv` para
  las que superan ese tamaño (Cargos Bis, Matrícula por edad), más `aprender_desempenos.xlsx`
  (niveles de desempeño por año/área/jurisdicción) y `diccionario_maestro.xlsx`.

Colab te pedirá autorizar el acceso a Drive. Fuera de Colab, la celda no hace nada.

In [11]:
import shutil
EXCEL_MAX = 200_000   # umbral .xlsx vs .csv (tablas mayores -> CSV: rápido y sin límite de filas)

def exportar_resultados(origen, destino):
    p_parquet = os.path.join(destino, 'parquet'); p_excel = os.path.join(destino, 'excel')
    os.makedirs(p_parquet, exist_ok=True); os.makedirs(p_excel, exist_ok=True)
    # 1) Parquet (para notebook 01)
    for sub in ['ra', 'aprender_long']:
        dst = os.path.join(p_parquet, sub)
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(os.path.join(origen, sub), dst)
    for f in ['diccionario_ra.parquet', 'diccionario_aprender.parquet']:
        shutil.copy2(os.path.join(origen, f), os.path.join(p_parquet, f))
    shutil.copy2(os.path.join(origen, 'catalogo_archivos.csv'), os.path.join(destino, 'catalogo_archivos.csv'))
    shutil.copy2(os.path.join(origen, 'diccionario_maestro.xlsx'), os.path.join(p_excel, 'diccionario_maestro.xlsx'))
    # 2) Excel/CSV legible — RA por base
    for fn in sorted(os.listdir(os.path.join(origen, 'ra'))):
        base = fn.replace('ra_', '').replace('.parquet', '')
        df = pd.read_parquet(os.path.join(origen, 'ra', fn))
        if len(df) <= EXCEL_MAX:
            df.to_excel(os.path.join(p_excel, f'ra_{base}.xlsx'), index=False)
        else:
            df.to_csv(os.path.join(p_excel, f'ra_{base}.csv'), index=False, encoding='utf-8-sig')
    # 3) Excel legible — APRENDER niveles de desempeño (subset)
    parts = []
    ald = os.path.join(origen, 'aprender_long')
    for anio_dir in sorted(os.listdir(ald)):
        d = pd.read_parquet(os.path.join(ald, anio_dir))
        d = d[d['tipo_variable'] == 'desempeño'].copy()
        d['anio'] = int(anio_dir.split('=')[1])
        parts.append(d)
    desemp = pd.concat(parts, ignore_index=True)
    desemp['nivel_desempeno'] = desemp['variable'].astype(str).str.split('_', n=1).str[1]
    desemp = desemp[['anio','cobertura','nivel','grado','area','jurisdiccion','departamento',
                     'sector','ambito','nivel_desempeno','valor']].sort_values(
                     ['anio','area','jurisdiccion','departamento','sector'])
    dest_d = os.path.join(p_excel, 'aprender_desempenos')
    (desemp.to_excel(dest_d + '.xlsx', index=False) if len(desemp) <= EXCEL_MAX
     else desemp.to_csv(dest_d + '.csv', index=False, encoding='utf-8-sig'))
    return destino

if in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    DESTINO = '/content/drive/MyDrive/pruebas_aprender'
    os.makedirs(DESTINO, exist_ok=True)
    print('Guardando en', DESTINO, '...')
    exportar_resultados(OUT, DESTINO)
    print('Listo. Resultados en Mi unidad/pruebas_aprender (parquet/ y excel/).')
else:
    print('Entorno local: se omite el guardado en Google Drive.')
    print("Para guardar localmente podés llamar: exportar_resultados(OUT, r'ruta/pruebas_aprender')")

Entorno local: se omite el guardado en Google Drive.
Para guardar localmente podés llamar: exportar_resultados(OUT, r'ruta/pruebas_aprender')


---
## Cómo usar los datos consolidados

```python
import pandas as pd

# Diccionario: qué significa cada variable
dic = pd.read_excel('datos_consolidados/diccionario_maestro.xlsx', sheet_name=None)

# RA (ancho): matrícula de todos los años
mat = pd.read_parquet('datos_consolidados/ra/ra_matricula.parquet')

# APRENDER (largo): leer todo, o filtrar por año (partición) para eficiencia
ap_all  = pd.read_parquet('datos_consolidados/aprender_long')
ap_2024 = pd.read_parquet('datos_consolidados/aprender_long', filters=[('anio', '==', 2024)])

# Solo niveles de desempeño
desemp = ap_all[ap_all.tipo_variable == 'desempeño']
```

**Desde Google Drive (para el notebook 01, en Colab):**
```python
from google.colab import drive; drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/pruebas_aprender/parquet'
mat     = pd.read_parquet(f'{BASE}/ra/ra_matricula.parquet')
ap_2024 = pd.read_parquet(f'{BASE}/aprender_long', filters=[('anio', '==', 2024)])
```

**Notas de interpretación** (ver hoja `Notas` del diccionario):
- APRENDER: los valores son **conteos ponderados de estudiantes** (factor de expansión), no porcentajes.
- RA: **conteos absolutos**.
- `departamento` puede faltar en APRENDER (agregación a nivel provincia).
- Las preguntas de contexto se repiten entre las áreas (Lengua/Matemática/Solo CC) de un mismo operativo:
  para el cuestionario canónico, filtrar `area == 'Solo CC'`.